<a href="https://colab.research.google.com/github/NeilKapoor2/Predicting-Soccer-Player-Salaries/blob/main/Performance_Based_Salaries.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import kagglehub
import os

path = kagglehub.dataset_download("armaanmartins21/undervalued-football-players")

print("Path to dataset files:", path)

files = os.listdir(path)

print("Files in dataset folder:")
for file in files:
    print(file)

csv_file = [file for file in files if file.endswith(".csv")][0]

df = pd.read_csv(os.path.join(path, csv_file))

print("Dataset loaded successfully!")

print("Original dataset shape:")
print(df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("First 5 rows:")
print(df.head())

print("Missing values in each column:")
print(df.isnull().sum())

# Columns that will be used in the model
model_columns = [
    "MP",
    "Starts",
    "Min",
    "90s",
    "Gls",
    "Ast",
    "G+A",
    "xG",
    "xAG",
    "PrgC",
    "PrgP",
    "PrgR",
    "Annual USD"
]

# Remove rows only if they are missing one of the columns we need
original_rows = len(df)

df = df.dropna(subset=model_columns)

rows_removed = original_rows - len(df)

print("Rows before removing missing values:", original_rows)
print("Rows removed:", rows_removed)
print("Rows after removing missing values:", len(df))

print(df[[
    "Player",
    "MP",
    "Gls",
    "Ast",
    "xG",
    "xAG",
    "Annual USD"
]].head(10))

print("Cleaned dataset:")
print(df.head())

print("\nDataset shape:")
print(df.shape)

# Input features
X = df[[        # array of inputs
    "MP",
    "Starts",
    "Min",
    "90s",
    "Gls",
    "Ast",
    "G+A",
    "xG",
    "xAG",
    "PrgC",
    "PrgP",
    "PrgR"
]]

# Output (target)
y = df["Annual USD"]

print(X.head())
print(y.head())

100%|██████████| 289k/289k [00:00<00:00, 617kB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/armaanmartins21/undervalued-football-players/versions/1
Files in dataset folder:
dfAll.csv
Dataset loaded successfully!
Original dataset shape:
(2831, 54)

Column names:
['Rk_x', 'Player', 'Nation_x', 'Pos_x', 'Squad_x', 'Age_x', 'Weekly Wages', 'Annual Wages', 'Annual EUR', 'Annual GBP', 'Annual USD', 'Rk_y', 'Nation_y', 'Pos_y', 'Squad_y', 'Age_y', 'Born', 'MP', 'Starts', 'Min', '90s', 'Gls', 'Ast', 'G+A', 'G-PK', 'PK', 'PKatt', 'CrdY', 'CrdR', 'xG', 'npxG', 'xAG', 'npxG+xAG', 'PrgC', 'PrgP', 'PrgR', 'Gls.1', 'Ast.1', 'G+A.1', 'G-PK.1', 'G+A-PK', 'xG.1', 'xAG.1', 'xG+xAG', 'npxG.1', 'npxG+xAG.1', 'Matches', 'Predicted Salary', 'Ratio', 'League', 'Notes', 'Rk', 'k', 'log salary']
First 5 rows:
   Rk_x           Player Nation_x  Pos_x       Squad_x  Age_x  \
0   1.0     Lionel Messi   ar ARG  FW,MF   Inter Miami     37   
1   2.0    Son Heung-min   kr KOR     FW          LAFC     32   
2   3.0  Sergio Busquets   es ESP     MF   Int

In [12]:
from sklearn.model_selection import train_test_split    # splits dataset into training, validation, and testing cases

# First split: 80% training/validation and 20% testing
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=0.2      # 20% of the data for testing
)

# Second split: 80% training and 20% validation
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=0.2      # 20% of the remaining 80% = 16% of total data
)

print("Training data:")
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)

print("\nValidation data:")
print("X_val:", X_val.shape)
print("y_val:", y_val.shape)

print("\nTesting data:")
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

Training data:
X_train: (1811, 12)
y_train: (1811,)

Validation data:
X_val: (453, 12)
y_val: (453,)

Testing data:
X_test: (567, 12)
y_test: (567,)


In [14]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# Possible hyperparameter values
fit_intercept_values = [True, False]

best_mse = float("inf")
best_params = None

# Try every combination of hyperparameters
for fit_intercept in fit_intercept_values:

    lr_model = LinearRegression(
        fit_intercept=fit_intercept
    )

    # Train using ONLY the training data
    lr_model.fit(X_train, y_train)

    # Make predictions using ONLY the validation data
    val_predictions = lr_model.predict(X_val)

    # Calculate validation MSE
    val_mse = mean_squared_error(y_val, val_predictions)

    print("fit_intercept:", fit_intercept)
    print("Validation MSE:", val_mse)
    print()

    # Keep the best hyperparameters
    if val_mse < best_mse:
        best_mse = val_mse
        best_params = {
            "fit_intercept": fit_intercept
        }

print("Best Linear Regression parameters:")
print(best_params)

print("\nBest Validation MSE:")
print(best_mse)

fit_intercept: True
Validation MSE: 15838519445076.027

fit_intercept: False
Validation MSE: 17826529165116.707

Best Linear Regression parameters:
{'fit_intercept': True}

Best Validation MSE:
15838519445076.027


In [15]:
# Create the Linear Regression model using the best hyperparameters
lr_model = LinearRegression(
    **best_params
)

# Train the model using the training data
lr_model.fit(X_train, y_train)

# Make predictions on the validation set
val_predictions_lr = lr_model.predict(X_val)

print("First 10 validation predictions:")
print(val_predictions_lr[:10])

First 10 validation predictions:
[1702165.16680985 3060718.21955293 2454784.52417364 6575629.43804342
 3015582.31494949 2515725.24010598 4315267.33753975 2445394.48360066
 1721025.67029934 3159624.75415023]


In [16]:
# Make predictions on the test set
test_predictions_lr = lr_model.predict(X_test)

# Calculate test MSE
mse_test_lr = mean_squared_error(y_test, test_predictions_lr)

print("Linear Regression MSE (Test):", mse_test_lr)

Linear Regression MSE (Test): 18323710225252.02


In [4]:
from sklearn.metrics import mean_absolute_error, r2_score

# Calculate Mean Absolute Error
mae = mean_absolute_error(y_test, predictions)

# Calculate R² Score
r2 = r2_score(y_test, predictions)

print("Mean Absolute Error:", mae)
print("R² Score:", r2)

Mean Absolute Error: 2763375.1257891636
R² Score: 0.13742122414011904


In [5]:
for i in range(len(X_test)):
    player = df.loc[X_test.index[i], "Player"]

    actual = y_test.iloc[i]
    predicted = predictions[i]

    percent_error = abs((actual - predicted) / actual) * 100

    print(f"{player} | Actual: ${actual:,.2f} | Predicted: ${predicted:,.2f} | Error: {percent_error:.2f}%")

Hugo Guillamón | Actual: $1,979,194.00 | Predicted: $1,882,055.42 | Error: 4.91%
Giuliano Simeone | Actual: $515,415.00 | Predicted: $930,840.16 | Error: 80.60%
Joelinton | Actual: $10,219,571.00 | Predicted: $4,561,026.21 | Error: 55.37%
Stefan Bajcetic | Actual: $2,824,016.00 | Predicted: $3,048,035.54 | Error: 7.93%
Jacob Ondrejka | Actual: $1,076,014.00 | Predicted: $3,439,511.80 | Error: 219.65%
Aleksei Miranchuk | Actual: $3,600,000.00 | Predicted: $6,658,579.28 | Error: 84.96%
Alex Iwobi | Actual: $5,450,438.00 | Predicted: $7,714,387.58 | Error: 41.54%
Pedro Porro | Actual: $5,791,090.00 | Predicted: $5,109,840.52 | Error: 11.76%
Aiden O'Neill | Actual: $540,000.00 | Predicted: $3,251,466.69 | Error: 502.12%
Gilberto Flores | Actual: $364,992.00 | Predicted: $2,938,005.98 | Error: 704.95%
Jeremie Boga | Actual: $4,628,019.00 | Predicted: $3,369,667.62 | Error: 27.19%
Stole Dimitrievski | Actual: $1,278,229.00 | Predicted: $3,404,580.61 | Error: 166.35%
Leandro Trossard | Actual

In [40]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error

# Possible hyperparameter values
max_depth_values = [3, 5, 10, 15, 20, None]
min_samples_split_values = [2, 5, 10]
min_samples_leaf_values = [1, 2, 5]

best_mse = float("inf")
best_params = None

# Try every combination of hyperparameters
for max_depth in max_depth_values:
    for min_samples_split in min_samples_split_values:
        for min_samples_leaf in min_samples_leaf_values:

            tree_model = DecisionTreeRegressor(
                max_depth=max_depth,
                min_samples_split=min_samples_split,
                min_samples_leaf=min_samples_leaf,
                random_state=42
            )

            # Train using ONLY the training data
            tree_model.fit(X_train, y_train)

            # Make predictions using ONLY the validation data
            val_predictions_tree = tree_model.predict(X_val)

            # Calculate validation MSE
            val_mse = mean_squared_error(y_val, val_predictions_tree)

            # Keep the best hyperparameters
            if val_mse < best_mse:
                best_mse = val_mse
                best_params = {
                    "max_depth": max_depth,
                    "min_samples_split": min_samples_split,
                    "min_samples_leaf": min_samples_leaf
                }

print("Best Decision Tree parameters:")
print(best_params)

print("\nBest Validation MSE:")
print(best_mse)

Best Decision Tree parameters:
{'max_depth': 5, 'min_samples_split': 2, 'min_samples_leaf': 2}

Best Validation MSE:
16357971198473.492


In [41]:
# Create the Decision Tree using the best hyperparameters
tree_model = DecisionTreeRegressor(
    **best_params,
    random_state=42
)

# Train the best model using the training data
tree_model.fit(X_train, y_train)

# Make predictions on the validation set
val_predictions_tree = tree_model.predict(X_val)

print("First 10 validation predictions:")
print(val_predictions_tree[:10])

First 10 validation predictions:
[3659554.20481928 3659554.20481928 2692196.81034483 4785726.28571429
 2077015.55       2435885.93419355 7119193.76666667 2435885.93419355
 4712125.46153846 2077015.55      ]


In [42]:
# Make predictions on the test set
test_predictions_tree = tree_model.predict(X_test)

# Calculate test MSE
mse_test_tree = mean_squared_error(y_test, test_predictions_tree)

print("Decision Tree MSE (Test):", mse_test_tree)

Decision Tree MSE (Test): 20121883444568.42


In [43]:
for i in range(len(X_test)):
    player = df.loc[X_test.index[i], "Player"]

    actual = y_test.iloc[i]
    predicted = test_predictions_tree[i]

    percent_error = abs((actual - predicted) / actual) * 100

    print(
        f"{player} | "
        f"Actual: ${actual:,.2f} | "
        f"Predicted: ${predicted:,.2f} | "
        f"Error: {percent_error:.2f}%"
    )

Pedro | Actual: $3,262,754.00 | Predicted: $2,692,196.81 | Error: 17.49%
Pablo Maffeo | Actual: $1,010,213.00 | Predicted: $2,435,885.93 | Error: 141.13%
Nikola Vlašić | Actual: $2,961,932.00 | Predicted: $2,692,196.81 | Error: 9.11%
Luke Shaw | Actual: $10,219,571.00 | Predicted: $3,659,554.20 | Error: 64.19%
Rafael Santos | Actual: $375,000.00 | Predicted: $2,435,885.93 | Error: 549.57%
Ulisses Garcia | Actual: $2,314,010.00 | Predicted: $2,435,885.93 | Error: 5.27%
Mauricio Pineda | Actual: $355,012.00 | Predicted: $2,435,885.93 | Error: 586.14%
Kevin Schade | Actual: $681,305.00 | Predicted: $4,712,125.46 | Error: 591.63%
Warren Kamanzi | Actual: $636,353.00 | Predicted: $2,435,885.93 | Error: 282.79%
Dejan Kulusevski | Actual: $7,494,352.00 | Predicted: $4,712,125.46 | Error: 37.12%
Noni Madueke | Actual: $10,219,571.00 | Predicted: $4,785,726.29 | Error: 53.17%
Abdou Harroui | Actual: $890,894.00 | Predicted: $3,659,554.20 | Error: 310.77%
Francesco Acerbi | Actual: $3,254,140.00

In [9]:
from sklearn.metrics import mean_squared_error

# Linear Regression predictions
train_pred_lr = lr_model.predict(X_train)
test_pred_lr = lr_model.predict(X_test)

mse_train_lr = mean_squared_error(y_train, train_pred_lr)
mse_test_lr = mean_squared_error(y_test, test_pred_lr)

print("Linear Regression MSE (Train):", mse_train_lr)
print("Linear Regression MSE (Test):", mse_test_lr)


# Decision Tree predictions
train_pred_tree = tree_model.predict(X_train)
test_pred_tree = tree_model.predict(X_test)

mse_train_tree = mean_squared_error(y_train, train_pred_tree)
mse_test_tree = mean_squared_error(y_test, test_pred_tree)

print("Decision Tree MSE (Train):", mse_train_tree)
print("Decision Tree MSE (Test):", mse_test_tree)

Linear Regression MSE (Train): 16911863340276.021
Linear Regression MSE (Test): 16386625913292.861
Decision Tree MSE (Train): 245279163274.58646
Decision Tree MSE (Test): 41288088341577.26


In [18]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

# Possible hyperparameter values
n_estimators_values = [50, 100, 200]
max_depth_values = [5, 10, 15, None]
min_samples_split_values = [2, 5, 10]
min_samples_leaf_values = [1, 2, 5]

best_mse = float("inf")
best_params = None

# Try every combination of hyperparameters
for n_estimators in n_estimators_values:
    for max_depth in max_depth_values:
        for min_samples_split in min_samples_split_values:
            for min_samples_leaf in min_samples_leaf_values:

                rf_model = RandomForestRegressor(
                    n_estimators=n_estimators,
                    max_depth=max_depth,
                    min_samples_split=min_samples_split,
                    min_samples_leaf=min_samples_leaf,
                    random_state=42
                )

                # Train using ONLY the training data
                rf_model.fit(X_train, y_train)

                # Make predictions using ONLY the validation data
                val_predictions = rf_model.predict(X_val)

                # Calculate validation MSE
                val_mse = mean_squared_error(y_val, val_predictions)

                # Keep the best hyperparameters
                if val_mse < best_mse:
                    best_mse = val_mse
                    best_params = {
                        "n_estimators": n_estimators,
                        "max_depth": max_depth,
                        "min_samples_split": min_samples_split,
                        "min_samples_leaf": min_samples_leaf
                    }

print("Best Random Forest parameters:")
print(best_params)

print("\nBest Validation MSE:")
print(best_mse)

Best Random Forest parameters:
{'n_estimators': 200, 'max_depth': 5, 'min_samples_split': 2, 'min_samples_leaf': 1}

Best Validation MSE:
15267716378460.174


In [19]:
# Create the Random Forest using the best hyperparameters
rf_model = RandomForestRegressor(
    **best_params,
    random_state=42
)

# Train the model using the training data
rf_model.fit(X_train, y_train)

# Make predictions on the validation set
val_predictions_rf = rf_model.predict(X_val)

print("First 10 validation predictions:")
print(val_predictions_rf[:10])

First 10 validation predictions:
[2551324.26253441 3050865.34405862 2721117.45152697 5207636.74948177
 2383434.64690405 2878576.82573113 5454173.38123967 2040726.08687496
 2412758.15749294 2455220.21690653]


In [20]:
# Make predictions on the test set
test_predictions_rf = rf_model.predict(X_test)

# Calculate test MSE
mse_test_rf = mean_squared_error(y_test, test_predictions_rf)

print("Random Forest MSE (Test):", mse_test_rf)

Random Forest MSE (Test): 16207949663502.473


In [21]:
for i in range(len(X_test)):
    player = df.loc[X_test.index[i], "Player"]

    actual = y_test.iloc[i]
    predicted = test_predictions_rf[i]

    percent_error = abs((actual - predicted) / actual) * 100

    print(f"{player} | Actual: ${actual:,.2f} | Predicted: ${predicted:,.2f} | Error: {percent_error:.2f}%")

Pedro | Actual: $3,262,754.00 | Predicted: $3,699,919.23 | Error: 13.40%
Pablo Maffeo | Actual: $1,010,213.00 | Predicted: $2,539,932.38 | Error: 151.43%
Nikola Vlašić | Actual: $2,961,932.00 | Predicted: $2,862,868.13 | Error: 3.34%
Luke Shaw | Actual: $10,219,571.00 | Predicted: $2,993,354.76 | Error: 70.71%
Rafael Santos | Actual: $375,000.00 | Predicted: $2,318,474.29 | Error: 518.26%
Ulisses Garcia | Actual: $2,314,010.00 | Predicted: $2,246,511.66 | Error: 2.92%
Mauricio Pineda | Actual: $355,012.00 | Predicted: $2,134,331.16 | Error: 501.20%
Kevin Schade | Actual: $681,305.00 | Predicted: $3,967,337.69 | Error: 482.31%
Warren Kamanzi | Actual: $636,353.00 | Predicted: $2,069,168.67 | Error: 225.16%
Dejan Kulusevski | Actual: $7,494,352.00 | Predicted: $4,151,183.93 | Error: 44.61%
Noni Madueke | Actual: $10,219,571.00 | Predicted: $5,160,315.43 | Error: 49.51%
Abdou Harroui | Actual: $890,894.00 | Predicted: $3,873,870.21 | Error: 334.83%
Francesco Acerbi | Actual: $3,254,140.00

In [22]:
from sklearn.preprocessing import StandardScaler

# Create the scaler
scaler = StandardScaler()

# Fit the scaler ONLY on the training data
X_train_scaled = scaler.fit_transform(X_train)

# Use the same scaler to transform validation and testing data
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("Training data scaled:")
print(X_train_scaled[:5])

Training data scaled:
[[-1.07989647 -0.69054087 -0.70855246 -0.71269152  0.57800288 -0.66100597
   0.13560878  0.35174246 -0.36841291 -0.38325939 -0.79781369 -0.13420124]
 [ 1.03615602  1.53582723  1.57346008  1.57245195  2.25865026  1.73028811
   2.3248004   1.92119508  2.03491205  2.92172172  1.84613367  3.52266402]
 [ 0.57614461 -1.06160222 -0.95593396 -0.95787859 -0.26232081 -0.66100597
  -0.46144348  0.06347565 -0.69613904 -0.60359146 -0.85529081 -0.46808894]
 [ 0.85215145 -0.96883688 -0.43719567 -0.438082   -0.26232081 -0.18274716
  -0.26242606  0.06347565  0.6693865   0.24101482 -0.62538234  0.40637884]
 [-1.07989647 -0.59777553 -0.62463892 -0.62442417 -0.54242871 -0.66100597
  -0.6604609  -0.6091469  -0.75076006 -0.82392354 -0.98940408 -0.78607722]]


In [23]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error

# Possible hyperparameter values
n_neighbors_values = [3, 5, 10, 15, 20]
weights_values = ["uniform", "distance"]
p_values = [1, 2]

best_mse = float("inf")
best_params = None

# Try every combination of hyperparameters
for n_neighbors in n_neighbors_values:
    for weights in weights_values:
        for p in p_values:

            knn_model = KNeighborsRegressor(
                n_neighbors=n_neighbors,
                weights=weights,
                p=p
            )

            # Train using ONLY the training data
            knn_model.fit(X_train_scaled, y_train)

            # Make predictions using ONLY the validation data
            val_predictions = knn_model.predict(X_val_scaled)

            # Calculate validation MSE
            val_mse = mean_squared_error(y_val, val_predictions)

            # Keep the best hyperparameters
            if val_mse < best_mse:
                best_mse = val_mse
                best_params = {
                    "n_neighbors": n_neighbors,
                    "weights": weights,
                    "p": p
                }

print("Best KNN parameters:")
print(best_params)

print("\nBest Validation MSE:")
print(best_mse)

Best KNN parameters:
{'n_neighbors': 20, 'weights': 'uniform', 'p': 1}

Best Validation MSE:
15535043895847.695


In [24]:
# Create the KNN model using the best hyperparameters
knn_model = KNeighborsRegressor(
    **best_params
)

# Train the model using the training data
knn_model.fit(X_train_scaled, y_train)

# Make predictions on the validation set
val_predictions_knn = knn_model.predict(X_val_scaled)

print("First 10 validation predictions:")
print(val_predictions_knn[:10])

First 10 validation predictions:
[2771928.55 2314308.55 2748916.55 5096257.25 2731481.05 2191943.65
 5748054.85 2313168.45 2055960.15 1965502.2 ]


In [25]:
# Make predictions on the test set
test_predictions_knn = knn_model.predict(X_test_scaled)

# Calculate test MSE
mse_test_knn = mean_squared_error(y_test, test_predictions_knn)

print("KNN MSE (Test):", mse_test_knn)

KNN MSE (Test): 17572176549794.56


In [26]:
for i in range(len(X_test)):
    player = df.loc[X_test.index[i], "Player"]

    actual = y_test.iloc[i]
    predicted = test_predictions_knn[i]

    percent_error = abs((actual - predicted) / actual) * 100

    print(f"{player} | Actual: ${actual:,.2f} | Predicted: ${predicted:,.2f} | Error: {percent_error:.2f}%")

Pedro | Actual: $3,262,754.00 | Predicted: $3,471,554.15 | Error: 6.40%
Pablo Maffeo | Actual: $1,010,213.00 | Predicted: $1,860,549.65 | Error: 84.17%
Nikola Vlašić | Actual: $2,961,932.00 | Predicted: $1,824,680.70 | Error: 38.40%
Luke Shaw | Actual: $10,219,571.00 | Predicted: $3,903,379.80 | Error: 61.80%
Rafael Santos | Actual: $375,000.00 | Predicted: $1,327,973.75 | Error: 254.13%
Ulisses Garcia | Actual: $2,314,010.00 | Predicted: $2,309,769.95 | Error: 0.18%
Mauricio Pineda | Actual: $355,012.00 | Predicted: $2,454,943.95 | Error: 591.51%
Kevin Schade | Actual: $681,305.00 | Predicted: $4,786,789.60 | Error: 602.59%
Warren Kamanzi | Actual: $636,353.00 | Predicted: $1,265,287.85 | Error: 98.83%
Dejan Kulusevski | Actual: $7,494,352.00 | Predicted: $4,580,405.05 | Error: 38.88%
Noni Madueke | Actual: $10,219,571.00 | Predicted: $4,840,266.45 | Error: 52.64%
Abdou Harroui | Actual: $890,894.00 | Predicted: $4,509,493.40 | Error: 406.18%
Francesco Acerbi | Actual: $3,254,140.00 |

In [28]:
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error

# Possible hyperparameter values
hidden_layer_sizes_values = [
    (50,),
    (100,),
    (50, 50),
    (100, 50)
]

learning_rate_values = [0.001, 0.01]
alpha_values = [0.0001, 0.001, 0.01]

best_mse = float("inf")
best_params = None

# Try every combination of hyperparameters
for hidden_layer_sizes in hidden_layer_sizes_values:
    for learning_rate in learning_rate_values:
        for alpha in alpha_values:

            nn_model = MLPRegressor(
                hidden_layer_sizes=hidden_layer_sizes,
                learning_rate_init=learning_rate,
                alpha=alpha,
                max_iter=1000,
                random_state=42
            )

            # Train using ONLY the training data
            nn_model.fit(X_train_scaled, y_train)

            # Make predictions using ONLY the validation data
            val_predictions = nn_model.predict(X_val_scaled)

            # Calculate validation MSE
            val_mse = mean_squared_error(y_val, val_predictions)

            # Keep the best hyperparameters
            if val_mse < best_mse:
                best_mse = val_mse
                best_params = {
                    "hidden_layer_sizes": hidden_layer_sizes,
                    "learning_rate_init": learning_rate,
                    "alpha": alpha
                }

print("Best Neural Network parameters:")
print(best_params)

print("\nBest Validation MSE:")
print(best_mse)

/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perce

Best Neural Network parameters:
{'hidden_layer_sizes': (100, 50), 'learning_rate_init': 0.01, 'alpha': 0.0001}

Best Validation MSE:
15396586747390.893


In [29]:
# Create the Neural Network using the best hyperparameters
nn_model = MLPRegressor(
    **best_params,
    max_iter=1000,
    random_state=42
)

# Train the model using the training data
nn_model.fit(X_train_scaled, y_train)

# Make predictions on the validation set
val_predictions_nn = nn_model.predict(X_val_scaled)

print("First 10 validation predictions:")
print(val_predictions_nn[:10])

First 10 validation predictions:
[2267067.55946957 2695056.70442197 2779942.96252163 6389223.29104681
 3329203.61287936 2276160.65293604 4348722.40172895 1896820.45985647
 1208306.28138736 3518784.56423049]


In [30]:
# Make predictions on the test set
test_predictions_nn = nn_model.predict(X_test_scaled)

# Calculate test MSE
mse_test_nn = mean_squared_error(y_test, test_predictions_nn)

print("Neural Network MSE (Test):", mse_test_nn)

Neural Network MSE (Test): 17537921824350.645


In [31]:
for i in range(len(X_test)):
    player = df.loc[X_test.index[i], "Player"]

    actual = y_test.iloc[i]
    predicted = test_predictions_nn[i]

    percent_error = abs((actual - predicted) / actual) * 100

    print(f"{player} | Actual: ${actual:,.2f} | Predicted: ${predicted:,.2f} | Error: {percent_error:.2f}%")

Pedro | Actual: $3,262,754.00 | Predicted: $5,521,104.85 | Error: 69.22%
Pablo Maffeo | Actual: $1,010,213.00 | Predicted: $1,272,482.69 | Error: 25.96%
Nikola Vlašić | Actual: $2,961,932.00 | Predicted: $3,610,921.82 | Error: 21.91%
Luke Shaw | Actual: $10,219,571.00 | Predicted: $3,419,890.55 | Error: 66.54%
Rafael Santos | Actual: $375,000.00 | Predicted: $1,939,476.31 | Error: 417.19%
Ulisses Garcia | Actual: $2,314,010.00 | Predicted: $2,582,399.79 | Error: 11.60%
Mauricio Pineda | Actual: $355,012.00 | Predicted: $1,347,325.21 | Error: 279.52%
Kevin Schade | Actual: $681,305.00 | Predicted: $4,190,393.59 | Error: 515.05%
Warren Kamanzi | Actual: $636,353.00 | Predicted: $1,458,984.66 | Error: 129.27%
Dejan Kulusevski | Actual: $7,494,352.00 | Predicted: $5,222,879.50 | Error: 30.31%
Noni Madueke | Actual: $10,219,571.00 | Predicted: $6,583,547.06 | Error: 35.58%
Abdou Harroui | Actual: $890,894.00 | Predicted: $3,992,742.67 | Error: 348.17%
Francesco Acerbi | Actual: $3,254,140.0

In [36]:
from sklearn.metrics import mean_squared_error
import numpy as np

# Calculate validation MSE for each model
mse_val_lr = mean_squared_error(y_val, val_predictions_lr)
mse_val_tree = mean_squared_error(y_val, val_predictions_tree)
mse_val_rf = mean_squared_error(y_val, val_predictions_rf)
mse_val_knn = mean_squared_error(y_val, val_predictions_knn)
mse_val_nn = mean_squared_error(y_val, val_predictions_nn)

print("Linear Regression Validation MSE:", mse_val_lr)
print("Decision Tree Validation MSE:", mse_val_tree)
print("Random Forest Validation MSE:", mse_val_rf)
print("KNN Validation MSE:", mse_val_knn)
print("Neural Network Validation MSE:", mse_val_nn)

Linear Regression Validation MSE: 15838519445076.027
Decision Tree Validation MSE: 16357971198473.492
Random Forest Validation MSE: 15267716378460.174
KNN Validation MSE: 15535043895847.695
Neural Network Validation MSE: 15396586747390.893


In [37]:
# Calculate inverse-MSE scores
inverse_mse = np.array([
    1 / mse_val_lr,
    1 / mse_val_tree,
    1 / mse_val_rf,
    1 / mse_val_knn,
    1 / mse_val_nn
])

# Normalize the weights so they add up to 1
weights = inverse_mse / np.sum(inverse_mse)

weight_lr = weights[0]
weight_tree = weights[1]
weight_rf = weights[2]
weight_knn = weights[3]
weight_nn = weights[4]

print("Linear Regression weight:", weight_lr)
print("Decision Tree weight:", weight_tree)
print("Random Forest weight:", weight_rf)
print("KNN weight:", weight_knn)
print("Neural Network weight:", weight_nn)

print("\nTotal weight:", np.sum(weights))

Linear Regression weight: 0.19786819965936903
Decision Tree weight: 0.191584842022436
Random Forest weight: 0.20526575488974177
KNN weight: 0.20173353541052827
Neural Network weight: 0.20354766801792482

Total weight: 0.9999999999999999


In [44]:
# Create weighted average of the model predictions
weighted_ensemble_predictions = (
    weight_lr * test_predictions_lr +
    weight_tree * test_predictions_tree +
    weight_rf * test_predictions_rf +
    weight_knn * test_predictions_knn +
    weight_nn * test_predictions_nn
)

print("First 10 weighted ensemble predictions:")
print(weighted_ensemble_predictions[:10])

First 10 weighted ensemble predictions:
[4011412.45298256 1946548.80797842 3031471.33811794 3409721.10576033
 2002779.60028572 2455012.72826066 1868632.38326942 4269755.4227256
 1680107.00189275 4785510.10886391]


In [46]:
# Calculate the weighted ensemble MSE
weighted_ensemble_mse = mean_squared_error(
    y_test,
    weighted_ensemble_predictions
)

print("Weighted Ensemble Model MSE:", weighted_ensemble_mse)

Weighted Ensemble Model MSE: 16888570641499.514


In [47]:
# Calculate percent error for each prediction
weighted_ensemble_percent_error = np.abs(
    (y_test - weighted_ensemble_predictions) / y_test
) * 100

print(weighted_ensemble_percent_error.head(10))

# Calculate average percent error
avg_weighted_ensemble_percent_error = np.mean(
    weighted_ensemble_percent_error
)

print(
    f"Weighted Ensemble Average Percent Error: "
    f"{avg_weighted_ensemble_percent_error:.2f}%"
)

2253     22.945599
1910     92.686969
2267      2.347770
749      66.635379
373     434.074560
1488      6.093436
394     426.357527
1060    526.702493
1605    164.021228
800      36.145112
Name: Annual USD, dtype: float64
Weighted Ensemble Average Percent Error: 402.14%


In [48]:
# Calculate percent error for each prediction
weighted_ensemble_percent_error = np.abs(
    (y_test - weighted_ensemble_predictions) / y_test
) * 100

print(weighted_ensemble_percent_error.head(10))

# Calculate average percent error
avg_weighted_ensemble_percent_error = np.mean(
    weighted_ensemble_percent_error
)

print(
    f"Weighted Ensemble Average Percent Error: "
    f"{avg_weighted_ensemble_percent_error:.2f}%"
)

2253     22.945599
1910     92.686969
2267      2.347770
749      66.635379
373     434.074560
1488      6.093436
394     426.357527
1060    526.702493
1605    164.021228
800      36.145112
Name: Annual USD, dtype: float64
Weighted Ensemble Average Percent Error: 402.14%


In [49]:
# Use the validation predictions from each existing model as inputs
meta_X_train = np.column_stack([
    val_predictions_lr,
    val_predictions_tree,
    val_predictions_rf,
    val_predictions_knn,
    val_predictions_nn
])

# The actual validation salaries are the target
meta_y_train = y_val

print("Meta-model input shape:")
print(meta_X_train.shape)

print("\nFirst validation example:")
print(meta_X_train[0])

print("\nActual salary:")
print(meta_y_train.iloc[0])

Meta-model input shape:
(453, 5)

First validation example:
[1702165.16680985 3659554.20481928 2551324.26253441 2771928.55
 2267067.55946957]

Actual salary:
206166.0


In [50]:
from sklearn.linear_model import LinearRegression

# Create the meta-model
meta_model = LinearRegression()

# Train the meta-model using the validation predictions
meta_model.fit(meta_X_train, meta_y_train)

print("Meta-model trained successfully!")

Meta-model trained successfully!


In [51]:
# Use the test predictions from each existing model as inputs
meta_X_test = np.column_stack([
    test_predictions_lr,
    test_predictions_tree,
    test_predictions_rf,
    test_predictions_knn,
    test_predictions_nn
])

print("Meta-model test input shape:")
print(meta_X_test.shape)

print("\nFirst test example:")
print(meta_X_test[0])

Meta-model test input shape:
(567, 5)

First test example:
[4609253.69322238 2692196.81034483 3699919.23135549 3471554.15
 5521104.84569795]


In [52]:
# Make the final salary prediction
meta_predictions = meta_model.predict(meta_X_test)

print("First 10 final predictions:")
print(meta_predictions[:10])

First 10 final predictions:
[5107642.08699321 1019346.30318252 2707585.50025232 3856142.80747814
 1199086.19365267 2024183.08566973 1280598.87895121 5329727.27658872
  810942.33783842 5928985.69580758]


In [53]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Calculate evaluation metrics
meta_mse = mean_squared_error(y_test, meta_predictions)
meta_mae = mean_absolute_error(y_test, meta_predictions)
meta_r2 = r2_score(y_test, meta_predictions)

print("Meta-Model MSE:", meta_mse)
print("Meta-Model MAE:", meta_mae)
print("Meta-Model R² Score:", meta_r2)

Meta-Model MSE: 16019463797118.254
Meta-Model MAE: 2660245.44544273
Meta-Model R² Score: 0.24135305348172986


In [54]:
# Calculate percent error
meta_percent_error = np.abs(
    (y_test - meta_predictions) / y_test
) * 100

print(meta_percent_error.head(10))

# Calculate average percent error
avg_meta_percent_error = np.mean(meta_percent_error)

print(
    f"Meta-Model Average Percent Error: "
    f"{avg_meta_percent_error:.2f}%"
)

2253     56.543892
1910      0.904097
2267      8.587182
749      62.267077
373     219.756318
1488     12.524877
394     260.719885
1060    682.282132
1605     27.435926
800      20.887280
Name: Annual USD, dtype: float64
Meta-Model Average Percent Error: 371.01%


In [55]:
for i in range(len(X_test)):
    player = df.loc[X_test.index[i], "Player"]

    actual = y_test.iloc[i]
    predicted = meta_predictions[i]

    percent_error = abs((actual - predicted) / actual) * 100

    print(
        f"{player} | "
        f"Actual: ${actual:,.2f} | "
        f"Predicted: ${predicted:,.2f} | "
        f"Error: {percent_error:.2f}%"
    )

Pedro | Actual: $3,262,754.00 | Predicted: $5,107,642.09 | Error: 56.54%
Pablo Maffeo | Actual: $1,010,213.00 | Predicted: $1,019,346.30 | Error: 0.90%
Nikola Vlašić | Actual: $2,961,932.00 | Predicted: $2,707,585.50 | Error: 8.59%
Luke Shaw | Actual: $10,219,571.00 | Predicted: $3,856,142.81 | Error: 62.27%
Rafael Santos | Actual: $375,000.00 | Predicted: $1,199,086.19 | Error: 219.76%
Ulisses Garcia | Actual: $2,314,010.00 | Predicted: $2,024,183.09 | Error: 12.52%
Mauricio Pineda | Actual: $355,012.00 | Predicted: $1,280,598.88 | Error: 260.72%
Kevin Schade | Actual: $681,305.00 | Predicted: $5,329,727.28 | Error: 682.28%
Warren Kamanzi | Actual: $636,353.00 | Predicted: $810,942.34 | Error: 27.44%
Dejan Kulusevski | Actual: $7,494,352.00 | Predicted: $5,928,985.70 | Error: 20.89%
Noni Madueke | Actual: $10,219,571.00 | Predicted: $7,345,391.12 | Error: 28.12%
Abdou Harroui | Actual: $890,894.00 | Predicted: $4,693,322.24 | Error: 426.81%
Francesco Acerbi | Actual: $3,254,140.00 | P

In [56]:
model_names = [
    "Linear Regression",
    "Decision Tree",
    "Random Forest",
    "KNN",
    "Neural Network"
]

print("Meta-model coefficients:")

for name, coefficient in zip(model_names, meta_model.coef_):
    print(f"{name}: {coefficient:.4f}")

print("\nMeta-model intercept:")
print(meta_model.intercept_)

Meta-model coefficients:
Linear Regression: -0.1076
Decision Tree: 0.2513
Random Forest: 0.3055
KNN: 0.4316
Neural Network: 0.7753

Meta-model intercept:
-1981963.3798123621


In [57]:
# Use the validation predictions from each existing model as inputs
meta_X_train = np.column_stack([
    val_predictions_lr,
    val_predictions_tree,
    val_predictions_rf,
    val_predictions_knn,
    val_predictions_nn
])

# The actual validation salaries are the target
meta_y_train = y_val

print("Meta-model training input shape:")
print(meta_X_train.shape)

print("\nFirst validation example:")
print(meta_X_train[0])

print("\nActual salary:")
print(meta_y_train.iloc[0])

Meta-model training input shape:
(453, 5)

First validation example:
[1702165.16680985 3659554.20481928 2551324.26253441 2771928.55
 2267067.55946957]

Actual salary:
206166.0


In [58]:
from sklearn.preprocessing import StandardScaler

# Create the scaler
meta_scaler = StandardScaler()

# Fit the scaler using the validation predictions
meta_X_train_scaled = meta_scaler.fit_transform(meta_X_train)

print("Scaled meta-model inputs:")
print(meta_X_train_scaled[:5])

Scaled meta-model inputs:
[[-1.03737765  0.27410125 -0.39875633 -0.01810127 -0.67175315]
 [-0.08365248  0.27410125 -0.02599817 -0.35550562 -0.37269958]
 [-0.50902727 -0.26675821 -0.27205644 -0.03506808 -0.31338607]
 [ 2.38386932  0.90375562  1.58338726  1.69563198  2.20856613]
 [-0.11533858 -0.61071237 -0.52403576 -0.04792331  0.07040487]]


In [59]:
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error

# Possible hyperparameter values
hidden_layer_sizes_values = [
    (10,),
    (20,),
    (10, 10),
    (20, 10)
]

learning_rate_values = [0.001, 0.01]
alpha_values = [0.0001, 0.001, 0.01]

best_mse = float("inf")
best_params = None

# Try every combination of hyperparameters
for hidden_layer_sizes in hidden_layer_sizes_values:
    for learning_rate in learning_rate_values:
        for alpha in alpha_values:

            meta_nn_model = MLPRegressor(
                hidden_layer_sizes=hidden_layer_sizes,
                learning_rate_init=learning_rate,
                alpha=alpha,
                max_iter=3000,
                early_stopping=True,
                validation_fraction=0.1,
                n_iter_no_change=20,
                random_state=42
            )

            # Train using the validation predictions
            meta_nn_model.fit(meta_X_train_scaled, meta_y_train)

            # Make predictions
            meta_val_predictions = meta_nn_model.predict(meta_X_train_scaled)

            # Calculate MSE
            meta_val_mse = mean_squared_error(
                meta_y_train,
                meta_val_predictions
            )

            print(
                "Hidden layers:", hidden_layer_sizes,
                "| Learning rate:", learning_rate,
                "| Alpha:", alpha,
                "| MSE:", meta_val_mse
            )

            # Keep the best hyperparameters
            if meta_val_mse < best_mse:
                best_mse = meta_val_mse
                best_params = {
                    "hidden_layer_sizes": hidden_layer_sizes,
                    "learning_rate_init": learning_rate,
                    "alpha": alpha
                }

print("\nBest Meta Neural Network parameters:")
print(best_params)

print("\nBest Meta Neural Network MSE:")
print(best_mse)

Hidden layers: (10,) | Learning rate: 0.001 | Alpha: 0.0001 | MSE: 28065958263099.47
Hidden layers: (10,) | Learning rate: 0.001 | Alpha: 0.001 | MSE: 28065958263099.47
Hidden layers: (10,) | Learning rate: 0.001 | Alpha: 0.01 | MSE: 28065958263099.47
Hidden layers: (10,) | Learning rate: 0.01 | Alpha: 0.0001 | MSE: 28065908379950.26
Hidden layers: (10,) | Learning rate: 0.01 | Alpha: 0.001 | MSE: 28065908379950.26
Hidden layers: (10,) | Learning rate: 0.01 | Alpha: 0.01 | MSE: 28065908379950.27
Hidden layers: (20,) | Learning rate: 0.001 | Alpha: 0.0001 | MSE: 28065953452180.09
Hidden layers: (20,) | Learning rate: 0.001 | Alpha: 0.001 | MSE: 28065953452180.09
Hidden layers: (20,) | Learning rate: 0.001 | Alpha: 0.01 | MSE: 28065953452180.09
Hidden layers: (20,) | Learning rate: 0.01 | Alpha: 0.0001 | MSE: 28065830362894.88
Hidden layers: (20,) | Learning rate: 0.01 | Alpha: 0.001 | MSE: 28065830362894.875
Hidden layers: (20,) | Learning rate: 0.01 | Alpha: 0.01 | MSE: 28065830362894.

In [60]:
# Create the Neural Network using the best hyperparameters
meta_nn_model = MLPRegressor(
    **best_params,
    max_iter=3000,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=20,
    random_state=42
)

# Train the best meta-model
meta_nn_model.fit(meta_X_train_scaled, meta_y_train)

print("Meta Neural Network trained successfully!")

Meta Neural Network trained successfully!


In [61]:
# Use the test predictions from each existing model
meta_X_test = np.column_stack([
    test_predictions_lr,
    test_predictions_tree,
    test_predictions_rf,
    test_predictions_knn,
    test_predictions_nn
])

# Scale using the SAME scaler used for training
meta_X_test_scaled = meta_scaler.transform(meta_X_test)

print("Meta-model test input shape:")
print(meta_X_test.shape)

print("\nFirst test example:")
print(meta_X_test[0])

Meta-model test input shape:
(567, 5)

First test example:
[4609253.69322238 2692196.81034483 3699919.23135549 3471554.15
 5521104.84569795]


In [62]:
# Make final salary predictions
meta_predictions = meta_nn_model.predict(meta_X_test_scaled)

print("First 10 final predictions:")
print(meta_predictions[:10])

First 10 final predictions:
[135.46248839  51.46801881  41.53336751  84.01149528  47.20746736
  33.74191435  56.20979583 170.30579859  58.63445613 213.3713276 ]


In [63]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Calculate evaluation metrics
meta_mse = mean_squared_error(y_test, meta_predictions)
meta_mae = mean_absolute_error(y_test, meta_predictions)
meta_r2 = r2_score(y_test, meta_predictions)

print("Meta Neural Network MSE:", meta_mse)
print("Meta Neural Network MAE:", meta_mae)
print("Meta Neural Network R² Score:", meta_r2)

Meta Neural Network MSE: 32683470825531.668
Meta Neural Network MAE: 3401200.8001366435
Meta Neural Network R² Score: -0.547818057922075


In [64]:
# Calculate percent error
meta_percent_error = np.abs(
    (y_test - meta_predictions) / y_test
) * 100

print(meta_percent_error.head(10))

# Calculate average percent error
avg_meta_percent_error = np.mean(meta_percent_error)

print(
    f"Meta Neural Network Average Percent Error: "
    f"{avg_meta_percent_error:.2f}%"
)

2253    99.995848
1910    99.994905
2267    99.998598
749     99.999178
373     99.987411
1488    99.998542
394     99.984167
1060    99.975003
1605    99.990786
800     99.997153
Name: Annual USD, dtype: float64
Meta Neural Network Average Percent Error: 99.99%


In [65]:
for i in range(len(X_test)):
    player = df.loc[X_test.index[i], "Player"]

    actual = y_test.iloc[i]
    predicted = meta_predictions[i]

    percent_error = abs((actual - predicted) / actual) * 100

    print(
        f"{player} | "
        f"Actual: ${actual:,.2f} | "
        f"Predicted: ${predicted:,.2f} | "
        f"Error: {percent_error:.2f}%"
    )

Pedro | Actual: $3,262,754.00 | Predicted: $135.46 | Error: 100.00%
Pablo Maffeo | Actual: $1,010,213.00 | Predicted: $51.47 | Error: 99.99%
Nikola Vlašić | Actual: $2,961,932.00 | Predicted: $41.53 | Error: 100.00%
Luke Shaw | Actual: $10,219,571.00 | Predicted: $84.01 | Error: 100.00%
Rafael Santos | Actual: $375,000.00 | Predicted: $47.21 | Error: 99.99%
Ulisses Garcia | Actual: $2,314,010.00 | Predicted: $33.74 | Error: 100.00%
Mauricio Pineda | Actual: $355,012.00 | Predicted: $56.21 | Error: 99.98%
Kevin Schade | Actual: $681,305.00 | Predicted: $170.31 | Error: 99.98%
Warren Kamanzi | Actual: $636,353.00 | Predicted: $58.63 | Error: 99.99%
Dejan Kulusevski | Actual: $7,494,352.00 | Predicted: $213.37 | Error: 100.00%
Noni Madueke | Actual: $10,219,571.00 | Predicted: $284.68 | Error: 100.00%
Abdou Harroui | Actual: $890,894.00 | Predicted: $150.07 | Error: 99.98%
Francesco Acerbi | Actual: $3,254,140.00 | Predicted: $128.23 | Error: 100.00%
Jayden Meghoma | Actual: $340,652.00 |

In [67]:
print("Model Performance Comparison")
print("-----------------------------------")

print("Linear Regression MSE:", mse_test_lr)
print("Decision Tree MSE:", mse_test_tree)
print("Random Forest MSE:", mse_test_rf)
print("KNN MSE:", mse_test_knn)
print("Neural Network MSE:", mse_test_nn)
print("Weighted Ensemble MSE:", weighted_ensemble_mse)
print("Meta Neural Network MSE:", meta_mse)

Model Performance Comparison
-----------------------------------
Linear Regression MSE: 18323710225252.02
Decision Tree MSE: 20121883444568.42
Random Forest MSE: 16207949663502.473
KNN MSE: 17572176549794.56
Neural Network MSE: 17537921824350.645
Weighted Ensemble MSE: 16888570641499.514
Meta Neural Network MSE: 32683470825531.668


In [68]:
model_mse = {
    "Linear Regression": mse_test_lr,
    "Decision Tree": mse_test_tree,
    "Random Forest": mse_test_rf,
    "KNN": mse_test_knn,
    "Neural Network": mse_test_nn,
    "Weighted Ensemble": weighted_ensemble_mse,
    "Meta Neural Network": meta_mse
}

best_model = min(model_mse, key=model_mse.get)
best_mse = model_mse[best_model]

print("Best Model:", best_model)
print("Best Test MSE:", best_mse)

Best Model: Random Forest
Best Test MSE: 16207949663502.473


In [69]:
# Calculate average percent error for each model

percent_error_lr = np.mean(
    np.abs((y_test - test_predictions_lr) / y_test) * 100
)

percent_error_tree = np.mean(
    np.abs((y_test - test_predictions_tree) / y_test) * 100
)

percent_error_rf = np.mean(
    np.abs((y_test - test_predictions_rf) / y_test) * 100
)

percent_error_knn = np.mean(
    np.abs((y_test - test_predictions_knn) / y_test) * 100
)

percent_error_nn = np.mean(
    np.abs((y_test - test_predictions_nn) / y_test) * 100
)

percent_error_ensemble = np.mean(
    np.abs((y_test - weighted_ensemble_predictions) / y_test) * 100
)

percent_error_meta = np.mean(
    np.abs((y_test - meta_predictions) / y_test) * 100
)

print("Linear Regression:", percent_error_lr)
print("Decision Tree:", percent_error_tree)
print("Random Forest:", percent_error_rf)
print("KNN:", percent_error_knn)
print("Neural Network:", percent_error_nn)
print("Weighted Ensemble:", percent_error_ensemble)
print("Meta Neural Network:", percent_error_meta)

Linear Regression: 422.2847266164849
Decision Tree: 425.9567704274422
Random Forest: 390.7814220082545
KNN: 357.5933926459846
Neural Network: 424.3064961826129
Weighted Ensemble: 402.1391009051858
Meta Neural Network: 99.98904313434534


In [70]:
model_percent_error = {
    "Linear Regression": percent_error_lr,
    "Decision Tree": percent_error_tree,
    "Random Forest": percent_error_rf,
    "KNN": percent_error_knn,
    "Neural Network": percent_error_nn,
    "Weighted Ensemble": percent_error_ensemble,
    "Meta Neural Network": percent_error_meta
}

best_percent_model = min(
    model_percent_error,
    key=model_percent_error.get
)

print("Best Model by Average Percent Error:", best_percent_model)
print(
    "Average Percent Error:",
    model_percent_error[best_percent_model]
)

Best Model by Average Percent Error: Meta Neural Network
Average Percent Error: 99.98904313434534
